In [1]:
from Bagging_for_LID.RunningEstimators.BaseEstimators import *
from Bagging_for_LID.run_files.final_tasks import *
from Bagging_for_LID.experiment_class import *
from Bagging_for_LID.RunningEstimators.Running2 import *
from Bagging_for_LID.Plotting.Plots.VariableInteraction import *
import skdim
from matplotlib import pyplot as plt

In [2]:
def threshold_testing_function(K=10, load1=False, load2=False):
    k_prog = [5+k for k in range(K-5)]
    sr_prog = [k/K for k in k_prog]
    param_dicts_none = {'dataset_name': dataset_name_strings,
           'n': 2500,
           'lid': None,
           'dim': None,
           'estimator_name': 'mle',
           'bagging_method': None,
           'submethod_0': '0',
           'submethod_error': 'log_diff',
           'k': K,
           'sr': 1,
           'Nbag': 10,
           'pre_smooth': False,
           'post_smooth': False,
           't': 1}
    param_dicts_bag = {'dataset_name': dataset_name_strings,
           'n': 2500,
           'lid': None,
           'dim': None,
           'estimator_name': 'mle',
           'bagging_method': 'bag',
           'submethod_0': '0',
           'submethod_error': 'log_diff',
           'k': k_prog,
           'sr': sr_prog,
           'Nbag': 10,
           'pre_smooth': False,
           'post_smooth': False,
           't': 1}
    
    exps_baseline = new_knn_dist_result_generator(param_dicts_none.copy(), multiprocess=True, load=load1, load_data=True, worker_count=5, save_name=f'knn_dists_res_pairs_baseline_k_{K}',directory=r'C:\Users\krp\PycharmProjects\FinalFixLIDGit\pkls')
    
    exps_bagged = new_knn_dist_result_generator(param_dicts_bag.copy(), multiprocess=True, load=load1, load_data=True, worker_count=5, save_name=f'knn_dists_res_pairs_bagged_k_{K}', directory=r'C:\Users\krp\PycharmProjects\FinalFixLIDGit\pkls')
    
    lid_results_baseline = new_result_generator(param_dicts_none.copy(), multiprocess=True, load=load2, load_data=True, worker_count=5, save_name=f'estimate_res_pairs_baseline_k_{K}', directory=r'C:\Users\krp\PycharmProjects\FinalFixLIDGit\pkls')
    
    lid_results_bagged = new_result_generator(param_dicts_bag.copy(), multiprocess=True, load=load2, load_data=True, worker_count=5, save_name=f'estimate_res_pairs_bagged_k_{K}', directory=r'C:\Users\krp\PycharmProjects\FinalFixLIDGit\pkls')
    
    plot_experiment_attr(
    exps_baseline+exps_bagged,
    x_param="sr",
    y_param="k",
    mode="difference",
    compare_param="bagging_method",
    compare_values=(None, "bag"),
    baseline_xy=(0.05, K),          # sr is ignored if it's not in _BASELINE_PARAMS
    diff_kind="difference",
    plot_kind="heatmap",
    value_attr="point_bag_avg_knn_dists",
    value_index=-1,
    save_prefix=f"distance_thresholds_baseline_k_{K}",
    formats=("pdf",),
    show=False,
    figsize=(25,25),
    base_fontsize=8,
    cmap="RdBu",
    )
    
    plot_experiment_heatmaps(
    lid_results_baseline+lid_results_bagged,
    x_param="sr",
    y_param="k",
    type="difference",
    baseline_overrides={"k": K},          # sr is ignored if it's not in _BASELINE_PARAMS
    save_prefix=f"distance_thresholds_baseline_k_{K}",
    formats=("pdf",),
    show=False,
    figsize=(25,25),
    base_fontsize=8,
    metrics=("mse", "bias2", "var"),
    label_every = 1,
    grid=True,
    log=True,
    inlog=False, 
    fig_title=f"k_sr_interaction_at_distance_thresholds_baseline_k_{K}",
    cmap="RdBu",
    
    )
    return exps_baseline+exps_bagged, lid_results_baseline+lid_results_bagged

In [3]:
distance_exps, lid_results_exps = threshold_testing_function(K=20, load1=True, load2=True)

<_io.BufferedReader name='C:\\Users\\krp\\PycharmProjects\\FinalFixLIDGit\\pkls\\knn_dists_res_pairs_baseline_k_20'>
<_io.BufferedReader name='C:\\Users\\krp\\PycharmProjects\\FinalFixLIDGit\\pkls\\knn_dists_res_pairs_bagged_k_20'>



KeyboardInterrupt



In [67]:
def threshold_comparrison_plot(exps_baseline, exps_bagged):
    
    distance_threshold_baseline = exps_baseline[0].point_bag_avg_knn_dists[-1] * np.ones(len(exps_bagged))
    distance_thresholds_std_baseline = np.std(exps_baseline[0].bag_avg_knn_dists[:,-1]) * np.ones(len(exps_bagged))

    distance_thresholds = []
    distance_thresholds_std = []
    for exp in exps_bagged:
        distance_thresholds.append(exp.point_bag_avg_knn_dists[-1])
        distance_thresholds_std.append(np.std(exp.bag_avg_knn_dists[:,-1]))
        
    distance_thresholds_array = np.array(distance_thresholds)
    
    distance_thresholds_std_array = np.array(distance_thresholds_array)
    
    x = np.array([x for x in range(len(exps_bagged))])
    
    fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=120)
    ax.plot(x, distance_threshold_baseline, linewidth=2, label="Threshold Baseline", color="blue")
    ax.plot(x, distance_thresholds_array, linewidth=2, label="Threshold Bagged", color="red")
    
    ax.fill_between(
        distance_threshold_baseline, distance_threshold_baseline-distance_thresholds_std_baseline, distance_threshold_baseline+distance_thresholds_std_baseline,
        alpha=0.3,
        color="blue",
        label=f"Threshold Baseline ± std",
    )
    
    ax.fill_between(
        distance_thresholds_array, distance_thresholds_array-distance_thresholds_std_array, distance_thresholds_array+distance_thresholds_std_array,
        alpha=0.4,
        color="red",
        label=f"Threshold Bagged ± std",
    )
    plt.close()
    return distance_threshold_baseline, distance_thresholds_array, distance_thresholds_std_array, distance_thresholds_std_baseline

In [37]:
K = 10

In [38]:
k_prog = [5+k for k in range(K-5)]
sr_prog = [k/K for k in k_prog]

In [39]:
param_dicts_none = {'dataset_name': dataset_name_strings[0],
           'n': 2500,
           'lid': None,
           'dim': None,
           'estimator_name': 'mle',
           'bagging_method': None,
           'submethod_0': '0',
           'submethod_error': 'log_diff',
           'k': K,
           'sr': 1,
           'Nbag': 10,
           'pre_smooth': False,
           'post_smooth': False,
           't': 1}

In [40]:
param_dicts_bag = {'dataset_name': dataset_name_strings[0],
           'n': 2500,
           'lid': None,
           'dim': None,
           'estimator_name': 'mle',
           'bagging_method': 'bag',
           'submethod_0': '0',
           'submethod_error': 'log_diff',
           'k': k_prog,
           'sr': sr_prog,
           'Nbag': 10,
           'pre_smooth': False,
           'post_smooth': False,
           't': 1}

In [41]:
exps_baseline = new_knn_dist_result_generator(param_dicts_none, multiprocess=True, load=False, load_data=False, worker_count=None, save_name='knn_dists_res_pairs', directory=r'C:\Users\krp\PycharmProjects\FinalFixLIDGit\pkls', expand_comb=True)

running experiments: 100%|██████████| 1/1 [00:01<00:00,  1.96s/exp]


In [42]:
exps_bagged = new_knn_dist_result_generator(param_dicts_bag, multiprocess=True, load=False, load_data=False, worker_count=None, save_name='knn_dists_res_pairs', directory=r'C:\Users\krp\PycharmProjects\FinalFixLIDGit\pkls', expand_comb=True)

running experiments: 100%|██████████| 5/5 [00:03<00:00,  1.34exp/s]


In [63]:
def threshold_comparrison_plot(exps_baseline, exps_bagged):
    
    distance_threshold_baseline = exps_baseline[0].point_bag_avg_knn_dists[-1] * np.ones(len(exps_bagged))
    distance_thresholds_std_baseline = np.std(exps_baseline[0].bag_avg_knn_dists[:,-1]) * np.ones(len(exps_bagged))

    distance_thresholds = []
    distance_thresholds_std = []
    for exp in exps_bagged:
        distance_thresholds.append(exp.point_bag_avg_knn_dists[-1])
        distance_thresholds_std.append(np.std(exp.bag_avg_knn_dists[:,-1]))
        
    distance_thresholds_array = np.array(distance_thresholds)
    
    distance_thresholds_std_array = np.array(distance_thresholds_array)
    
    x = np.array([x for x in range(len(exps_bagged))])
    
    fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=120)
    ax.plot(x, distance_threshold_baseline, linewidth=2, label="Threshold Baseline", color="blue")
    ax.plot(x, distance_thresholds_array, linewidth=2, label="Threshold Bagged", color="red")
    
    ax.fill_between(
        distance_threshold_baseline, distance_threshold_baseline-distance_thresholds_std_baseline, distance_threshold_baseline+distance_thresholds_std_baseline,
        alpha=0.3,
        color="blue",
        label=f"Threshold Baseline ± std",
    )
    
    ax.fill_between(
        distance_thresholds_array, distance_thresholds_array-distance_thresholds_std_array, distance_thresholds_array+distance_thresholds_std_array,
        alpha=0.4,
        color="red",
        label=f"Threshold Bagged ± std",
    )
    plt.close()
    return distance_threshold_baseline, distance_thresholds_array, distance_thresholds_std_array, distance_thresholds_std_baseline

In [64]:
distance_threshold_baseline, distance_thresholds_array, distance_thresholds_std_array, distance_thresholds_std_baseline = threshold_comparrison_plot(exps_baseline, exps_bagged)

In [66]:
np.mean((distance_threshold_baseline-distance_thresholds_array)**2)

3.002522132862163e-06

In [58]:
distance_threshold_baseline, distance_thresholds_array, distance_thresholds_std_array, distance_thresholds_std_baseline

(array([0.74129195, 0.74129195, 0.74129195, 0.74129195, 0.74129195]),
 array([0.73847801, 0.73924942, 0.73959193, 0.74128207, 0.74147154]),
 array([0.73847801, 0.73924942, 0.73959193, 0.74128207, 0.74147154]),
 array([0.02622735, 0.02622735, 0.02622735, 0.02622735, 0.02622735]))

In [50]:
distance_thresholds_array

array([0.73847801, 0.73924942, 0.73959193, 0.74128207, 0.74147154])

In [35]:
def knn_distances(X, k=10):
    dists, knnidx = skdim._commonfuncs.get_nn(X, k=k, n_jobs=1)
    return dists

In [36]:
def knn_distances_bagging(Q, X, n_bags=10, k=10, sampling_rate=None, w=None, indexuse=None, seed=42):
    rand_gen = np.random.RandomState(seed)
    def compute_distance_matrix(X):
        return squareform(pdist(X))

    def split_data_with_indices(X, n_bags=10):
        indices = np.random.permutation(len(X))
        if indexuse is not None:
            indices = np.intersect1d(indices, indexuse)
        split_indices = np.array_split(indices, n_bags)
        bags = [X[idx] for idx in split_indices]
        return bags, split_indices

    def sample_data_with_indices(X, n_bags=10, sampling_rate=0.8):
        n_samples = np.ceil(sampling_rate * len(X)).astype(int)
        indices = np.arange(len(X))
        bags = []
        selected_indices = []
        for _ in range(n_bags):
            chosen_idx = rand_gen.choice(indices, size=n_samples, replace=False)
            bags.append(X[chosen_idx])
            selected_indices.append(chosen_idx)
        return bags, selected_indices

    def k_smallest_nonzero_Q(distances, k=10, w=None):
        n = distances.shape[0]
        if w is not None:
            w = np.asarray(w)
            result_distances = []
            result_indices = []
            for q in range(n):
                nonzero_mask = distances[q] != 0
                valid_mask = nonzero_mask & (distances[q] <= w[q])
                dists = distances[q, valid_mask]
                inds = np.nonzero(valid_mask)[0]
                order = np.argsort(dists)
                result_distances.append(dists[order])
                result_indices.append(inds[order])
            return result_distances, result_indices
        else:
            result_distances = np.zeros((n, k))
            result_indices = np.zeros((n, k), dtype=int)
            for q in range(n):
                nonzero_mask = distances[q] != 0
                dists = distances[q, nonzero_mask]
                inds = np.nonzero(nonzero_mask)[0]
                if len(dists) >= k:
                    partition_indices = np.argpartition(dists, k)[:k]
                    sorted_indices = partition_indices[np.argsort(dists[partition_indices])]
                    result_distances[q, :] = dists[sorted_indices]
                    result_indices[q, :] = inds[sorted_indices]
                else:
                    raise ValueError("There are less nonzero distances than the given k")
            return result_distances, result_indices

    def k_smallest_distance_Q(distances, indices, k=10, w=None):
        considered_distances = distances[:, indices]
        smallest_distances, smallest_indices = k_smallest_nonzero_Q(considered_distances, k, w=w)
        if w is not None:
            smallest_indices = [indices[row_inds] for row_inds in smallest_indices]
        else:
            smallest_indices = indices[smallest_indices]
        return smallest_distances, smallest_indices

    n, m = Q.shape[0], Q.shape[1]
    distances = compute_distance_matrix(Q)
    
    if sampling_rate is None:
        bags, split_indices = split_data_with_indices(X, n_bags=n_bags)
    else:
        bags, split_indices = sample_data_with_indices(X, n_bags=n_bags, sampling_rate=sampling_rate)
        
    knn_distance_array = np.zeros((n, k, n_bags))
    
    for j in range(n_bags):
        smallest_distances, smallest_indices = k_smallest_distance_Q(distances=distances, indices=split_indices[j],
                                                                     k=k, w=w)
        knn_distance_array[:, :, j] = smallest_distances
        
    return knn_distance_array

In [37]:
dists = knn_distances(X, k=10)

In [38]:
dists

array([[0.621403  , 0.65076527, 0.65077228, ..., 0.78758709, 0.79002262,
        0.79155646],
       [0.4347216 , 0.58688338, 0.61103301, ..., 0.72243579, 0.73841221,
        0.73884256],
       [0.50464719, 0.5350495 , 0.61651698, ..., 0.70022663, 0.70478512,
        0.70588644],
       ...,
       [0.51802905, 0.54419168, 0.57163835, ..., 0.70849193, 0.70874027,
        0.7118905 ],
       [0.52035854, 0.62443337, 0.63068722, ..., 0.7061719 , 0.71131471,
        0.74315436],
       [0.43397171, 0.52568922, 0.52992785, ..., 0.69923575, 0.72538969,
        0.74143069]])

In [ ]:
dists